# 🚀 Databricks 기초 Hands-On Lab

## 📋 교육 목차

| 순서 | 주제 | 내용 |
|------|------|------|
| 1 | **환경 확인** | Spark 세션, 클러스터 정보 확인 |
| 2 | **DataFrame 기초** | DataFrame 생성, 스키마, 기본 연산 |
| 3 | **데이터 변환** | select, filter, groupBy, join, window 함수 |
| 4 | **Spark SQL** | Temp View, SQL 쿼리 실행 |
| 5 | **Delta Lake** | Delta 테이블 생성, CRUD, Time Travel |
| 6 | **Unity Catalog** | 카탈로그/스키마/테이블 관리 |

---
> **사전 준비**: 클러스터가 연결되어 있는지 확인하세요. Serverless Compute를 사용할 수도 있습니다.

## 1️⃣ 환경 확인

Databricks 노트북에서는 `SparkSession`이 자동으로 생성됩니다.  
`spark` 변수를 통해 바로 사용할 수 있습니다.

In [0]:
# Spark 버전 및 클러스터 정보 확인
print(f"Spark Version: {spark.version}")
print(f"App Name: {spark.conf.get('spark.app.name')}")
print(f"Master URL: {spark.conf.get('spark.master')}")

In [0]:
# 현재 사용자 및 카탈로그 확인
print(f"현재 카탈로그: {spark.catalog.currentCatalog()}")
print(f"현재 스키마: {spark.catalog.currentDatabase()}")
print(f"현재 사용자: {spark.sql('SELECT current_user()').collect()[0][0]}")

In [0]:
# 카탈로그 변경
spark.sql("USE CATALOG <카탈로그명>")

# 스키마 변경
spark.sql("USE <스키마명>")

## 2️⃣ DataFrame 기초

Spark의 핵심 데이터 구조인 **DataFrame**을 만들고 다루는 방법을 배울니다.

- DataFrame = 분산 환경에서 동작하는 테이블 형태의 데이터
- 불변(Immutable): 변환 시 새로운 DataFrame이 생성됨
- 지연 평가(Lazy Evaluation): Action이 호출될 때까지 실행되지 않음

In [0]:
# CSV 파일에서 DataFrame 생성
file_path = "/Volumes/edu260323/kgt/volume/youtube_top100.csv"

df_youtube = (
    spark.read.option("header", True).option("inferSchema", True).csv(file_path)
)

print(f"✅ CSV 파일 로드 완료: {file_path}")
display(df_youtube)

In [0]:
# 방법 2: 스키마를 명시적으로 지정하여 CSV 읽기
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
)

schema = StructType(
    [
        StructField("rank", IntegerType(), True),
        StructField("channel_name", StringType(), True),
        StructField("subscribers_in_millions", DoubleType(), True),
        StructField("primary_language", StringType(), True),
        StructField("category", StringType(), True),
        StructField("youTube_joined_date", StringType(), True),
        StructField("country", StringType(), True),
    ]
)

df_youtube_schema = spark.read.option("header", True).schema(schema).csv(file_path)

df_youtube_schema.printSchema()
display(df_youtube_schema)

In [0]:
# DataFrame 기본 정보 확인
print(f"행 수: {df_youtube_schema.count()}")
print(f"열 수: {len(df_youtube_schema.columns)}")
print(f"열 목록: {df_youtube_schema.columns}")
print(f"데이터 타입:")
df_youtube_schema.dtypes

In [0]:
# describe(): 수치형 컨럼의 통계 요약
display(df_youtube_schema.describe())

## 3️⃣ 데이터 변환 (Transformations)

DataFrame의 핵심 연산들을 실습합니다.

| 분류 | 메서드 | 설명 |
|------|--------|------|
| 선택 | `select()`, `drop()` | 컬럼 선택/제거 |
| 필터링 | `filter()`, `where()` | 조건별 행 필터링 |
| 집계 | `groupBy()`, `agg()` | 그룹별 집계 연산 |
| 정렬 | `orderBy()`, `sort()` | 데이터 정렬 |
| 결합 | `join()` | DataFrame 간 결합 |

In [0]:
from pyspark.sql.functions import col, lit, when, upper

# select: 특정 컨럼 선택
df_selected = df_youtube_schema.select(
    "rank", "channel_name", "subscribers_in_millions", "country"
)
display(df_selected)

# withColumn: 새 컨럼 추가
df_with_tier = df_youtube_schema.withColumn(
    "subscribers_grade",
    when(col("`subscribers_in_millions`") >= 150, "premium")
    .when(col("`subscribers_in_millions`") >= 100, "major")
    .otherwise("nomal"),
)
display(
    df_with_tier.select(
        "rank", "channel_name", "subscribers_in_millions", "subscribers_grade"
    )
)

In [0]:
# filter / where: 조건별 필터링
print("=== 구독자 1억 이상 채널 ===")
display(df_youtube_schema.filter(col("`subscribers_in_millions`") >= 100))

print("=== 영어 Entertainment 채널 ===")
display(
    df_youtube_schema.filter(
        (col("`primary_language`") == "English")
        & (col("category").contains("Entertainment"))
    )
)

print("=== 한국 또는 인도 채널 ===")
display(df_youtube_schema.filter(col("Country").isin("South Korea", "India")))

In [0]:
from pyspark.sql.functions import count, avg, sum, max, min, round

# groupBy + agg: 국가별 집계
df_country_stats = (
    df_youtube_schema.groupBy("country")
    .agg(
        count("*").alias("채널수"),
        round(avg("`subscribers_in_millions`"), 1).alias("평균구독자(M)"),
        max("`subscribers_in_millions`").alias("최대구독자(M)"),
        round(sum("`subscribers_in_millions`"), 0).alias("총구독자(M)"),
    )
    .orderBy("채널수", ascending=False)
)

display(df_country_stats)

In [0]:
# Join 실습을 위한 언어 정보 DataFrame 생성
lang_data = [
    ("English", "English", "Global"),
    ("Hindi", "Hindi", "Asia"),
    ("Korean", "Korean", "Asia"),
    ("Spanish", "Spanish", "Global"),
    ("Portuguese", "Portuguese", "South America"),
]

shema = "language_code string, language_name string, region string"

df_langs = spark.createDataFrame(
    lang_data, shema
)

# inner join: 언어 기준으로 결합
df_joined = df_youtube_schema.join(
    df_langs,
    df_youtube_schema["primary_language"] == df_langs["language_code"],
    how="inner",
).select(
    "rank",
    "channel_name",
    "subscribers_in_millions",
    "language_name",
    "region",
    "country",
)

display(df_joined)

## 4️⃣ Spark SQL

DataFrame을 **Temp View**로 등록하면 SQL로 직접 쿼리할 수 있습니다.

```
DataFrame → createOrReplaceTempView() → SQL 쿼리 → 결과 DataFrame
```

In [0]:
# DataFrame을 Temp View로 등록
df_youtube_schema.createOrReplaceTempView("youtube")

print("✅ Temp View 등록 완료: youtube")

In [0]:
%sql
-- SQL로 직접 쿼리 (Magic Command %sql 사용)
SELECT
  country,
  COUNT(*) AS chenal_count,
  ROUND(AVG(subscribers_in_millions), 1) AS subscribers_avg,
  MAX(subscribers_in_millions) AS subscribers_max
FROM
  youtube
GROUP BY
  country
ORDER BY
  chenal_count DESC

In [0]:
query = """
        SELECT 
            channel_name,
            subscribers_in_millions,
            country,
            category,
            CASE 
                WHEN cubscribers_in_millions >= 150 THEN premium
                WHEN cubscribers_in_millions >= 100 THEN major
                ELSE nomal
            END AS grade
        FROM youtube
        WHERE subscribers_in_millions >= 80
        ORDER BY subscribers_in_millions DESC
    """


df_top_channels = spark.sql(query)

display(df_top_channels)

## 5️⃣ Delta Lake

Delta Lake는 Databricks의 **기본 저장 포맷**입니다.

| 특징 | 설명 |
|------|------|
| **ACID 트랜잭션** | 데이터 일관성 보장 |
| **Time Travel** | 과거 버전 데이터 조회 |
| **Schema Evolution** | 스키마 변경 지원 |
| **MERGE (Upsert)** | INSERT + UPDATE 통합 연산 |
| **Time Travel** | 과거 버전 조회 및 변경 |

23번 cell 실행 전 ctrl + f 실행하여 `<catalog>.<schema>`를 테이블을 저장할 `카탈로그.스키마` 로 repalce all하여 모두 변경
ex) training.kgt

In [0]:
# Delta 테이블 생성 (DataFrame 저장)
df_youtube_schema.write.mode("overwrite").saveAsTable(
    "<catalog>.<schema>.youtube_top100"
)

print("✅ Delta 테이블 생성 완료")
display(spark.sql("SELECT * FROM <catalog>.<schema>.youtube_top100 ORDER BY Rank"))

In [0]:
%sql
-- Delta 테이블에 데이터 INSERT
INSERT INTO <catalog>.<schema>.youtube_top100 VALUES
    (101, 'NewChannel_KR', 75.0, 'Korean', 'Entertainment', 'January 1, 2020', 'South Korea'),
    (102, 'NewChannel_JP', 68.0, 'Japanese', 'Music', 'March 15, 2019', 'Japan');

SELECT * FROM <catalog>.<schema>.youtube_top100 
WHERE Rank >= 100 
ORDER BY Rank

In [0]:
%sql
-- Delta 테이블 UPDATE: 특정 채널의 구독자 수 수정
UPDATE <catalog>.<schema>.youtube_top100
SET subscribers_in_millions = subscribers_in_millions + 10
WHERE country = 'South Korea';

SELECT * FROM <catalog>.<schema>.youtube_top100
WHERE Country = 'South Korea'

In [0]:
%sql
-- Delta 테이블 DELETE
DELETE FROM <catalog>.<schema>.youtube_top100
WHERE Rank = 102;

SELECT * FROM <catalog>.<schema>.youtube_top100 WHERE Rank >= 100 ORDER BY Rank

In [0]:
%sql
-- MERGE (Upsert): 존재하면 UPDATE, 없으면 INSERT
MERGE INTO <catalog>.<schema>.youtube_top100 AS target
USING (
    SELECT 103 AS Rank, 'FreshChannel' AS channel_name, 90.0 AS subscribers_in_millions,
           'English' AS primary_language, 'Entertainment' AS category,
           'June 1, 2021' AS YouTube_joined_date, 'United States' AS country
    UNION ALL
    SELECT 101, 'NewChannel_KR', 85.0, 'Korean', 'Entertainment',
           'January 1, 2020', 'South Korea'  -- 기존 데이터 (UPDATE 됨)
) AS source
ON target.Rank = source.Rank
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;



SELECT * FROM <catalog>.<schema>.youtube_top100 WHERE Rank >= 100 ORDER BY Rank

In [0]:
%sql
-- Time Travel: 테이블 변경 이력 확인
DESCRIBE HISTORY <catalog>.<schema>.youtube_top100

In [0]:
%sql
-- Time Travel: 최초 버전(버전 0) 데이터 조회
SELECT * FROM <catalog>.<schema>.youtube_top100 VERSION AS OF 0
ORDER BY Rank

In [0]:
%sql
restore table <catalog>.<schema>.youtube_top100 VERSION AS OF 0

In [0]:
%sql
DESCRIBE EXTENDED <catalog>.<schema>.youtube_top100

In [0]:
dbutils.widgets.text("country_param", "", "국가 선택")

In [0]:
country = dbutils.widgets.get("country_param")
print(country)